# Final Project: FIFA World Cup 2026 Predictions

**Team Members:**   Maryam Sayeb, Eli Bryan, Jasmine Sidhu, Mark Vincent Baclagan 

**GitHub Repository Link:** https://github.com/sayebm/Group_Project


**Datasets:** https://www.kaggle.com/datasets/cashncarry/fifaworldranking/data
\
https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

## Introduction 

With the 2026 FIFA World Cup now just a few months away, anticipation for the tournament is growing rapidly worldwide. In British Columbia, competition feels even more exciting for residents this year, as Vancouver is one of the official host cities and will host seven matches at BC Place. The relevance of the World Cup in relation to Vancouver has motivated us to develop a prdeictive model to tend to the question; **Are we able to predict the nations with the largest chances to win the 2026 Word Cup with previous matches, and historical Word Cup results?**

## Overview

We are predicting the outcomes of the 2026 FIFA World Cup using two historical datasets of international football matches. These datasets contain information about past games, including team rankings, dates, match locations, and tournament details. We analyze patterns in previous matches to train a model that can accurately predict outcomes for upcoming matches.

We use supervised learning for this prediction because the dataset contains both input variables and known outcomes (which team won or lost). This allows the model to learn from past results and make predictions on upcoming matches.

The features used in this model represent key factors that can influence match outcomes. These include team rankings, which reflect the relative strength of each team, and the ranking difference, which captures how much stronger one team is compared to the other. We also include a binary variable to indicate whether the match was played at a neutral location, as this can affect team performance. These features provide information about team strength and match conditions, which are important for predicting match outcomes.

In [40]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**Load Results Dataset**

In [41]:
results_data = pd.read_csv("results.csv")
results_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


**Sorting the Results Dataset to Only Consider Matches from 2015 Onwards**

We restrict the dataset to matches from 2015 onwards to ensure that the data is more relevant to current team performance.

In [42]:
# Only considering matches starting after 2015
results_data['date'] = pd.to_datetime(results_data['date'])
results_filtered = results_data[results_data['date'] >= '2015-01-01']
results_filtered

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38380,2015-01-04,Bahrain,Jordan,1,0,Friendly,Ballarat,Australia,True
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38383,2015-01-04,South Africa,Zambia,1,0,Friendly,Johannesburg,South Africa,False
38384,2015-01-05,China PR,Oman,4,1,Friendly,Penrith,Australia,True
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


**Load rankings dataset and Convert the rank_date variable into a 'date' for python**

We load the rankings dataset and convert the rank_date variable into a date format so that Python can properly recognize and work with it as a time-based variable.

In [43]:
rankings_data = pd.read_csv("fifa_ranking-2024-06-20.csv")
rankings_data["year"] = pd.to_datetime(rankings_data["rank_date"]).dt.year
rankings_data


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date,year
0,140.0,Brunei Darussalam,BRU,2.00,0.00,140,AFC,1992-12-31,1992
1,33.0,Portugal,POR,38.00,0.00,33,UEFA,1992-12-31,1992
2,32.0,Zambia,ZAM,38.00,0.00,32,CAF,1992-12-31,1992
3,31.0,Greece,GRE,38.00,0.00,31,UEFA,1992-12-31,1992
4,30.0,Algeria,ALG,39.00,0.00,30,CAF,1992-12-31,1992
...,...,...,...,...,...,...,...,...,...
67467,137.0,Kuwait,KUW,1098.42,1085.46,-2,AFC,2024-06-20,2024
67468,136.0,Lithuania,LTU,1100.66,1095.23,-1,UEFA,2024-06-20,2024
67469,135.0,Malaysia,MAS,1107.58,1094.54,-3,AFC,2024-06-20,2024
67470,133.0,Solomon Islands,SOL,1111.02,1111.02,1,OFC,2024-06-20,2024


**Filter the Rankings Data**

We filter the rankings data to only include rankings from 2015 onwards to ensure the data reflects more recent team performance. And we keep only one ranking per country per year to reduce redundancy and simplify the dataset.

In [44]:
# Only considering rankings starting after 2015 and only have 1 observation per year per team
rankings_filtered = (
    rankings_data[rankings_data['year'] >= 2015]
    .groupby(['country_full', 'year'])
    .last()
    .reset_index()
)
rankings_filtered

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,Afghanistan,2015,150.0,AFG,194.00,168.00,-6,AFC,2015-12-03
1,Afghanistan,2016,146.0,AFG,189.00,189.00,-1,AFC,2016-12-22
2,Afghanistan,2017,148.0,AFG,181.00,181.00,1,AFC,2017-12-21
3,Afghanistan,2018,147.0,AFG,1068.00,1068.00,0,AFC,2018-12-20
4,Afghanistan,2019,149.0,AFG,1052.00,1052.00,0,AFC,2019-12-19
...,...,...,...,...,...,...,...,...,...
2101,Zimbabwe,2020,108.0,ZIM,1181.00,1181.00,0,CAF,2020-12-10
2102,Zimbabwe,2021,121.0,ZIM,1138.44,1138.44,0,CAF,2021-12-23
2103,Zimbabwe,2022,125.0,ZIM,1138.56,1138.56,0,CAF,2022-12-22
2104,Zimbabwe,2023,124.0,ZIM,1144.56,1144.56,0,CAF,2023-12-21


**Fix Spelling**

We fix spelling differences in country names across the datasets to ensure consistency. Some countries are recorded with slightly different names, which can cause issues. Standardizing these names ensures that each country is correctly matched.

In [45]:
name_mapping = {
    'USA':            'United States',
    'Korea Republic': 'South Korea',
    'Congo DR':       'DR Congo',
    "Cote d'Ivoire":  'Ivory Coast',
    'IR Iran':        'Iran',
}
rankings_data['country_full'] = rankings_data['country_full'].replace(name_mapping)
rankings_filtered['country_full'] = rankings_filtered['country_full'].replace(name_mapping)

**Teams Qualified for 2026 FIFA World Cup**

We identify the teams that have already qualified for the 2026 FIFA World Cup to focus our predictions on relevant participants.

In [46]:
world_cup_2026_teams = [
    # Host nations (auto-qualified)
    "United States", "Canada", "Mexico",
    # UEFA
    "Germany", "Portugal", "Spain", "France", "England", "Netherlands",
    "Belgium", "Austria", "Turkey", "Poland", "Serbia",
    # CONMEBOL
    "Argentina", "Colombia", "Uruguay", "Brazil", "Ecuador", "Paraguay",
    # CONCACAF
    "Panama", "Costa Rica", "Honduras", "Jamaica",
    # CAF
    "Morocco", "Egypt", "Senegal", "South Africa", "DR Congo",
    "Ivory Coast", "Nigeria", "Algeria", "Tunisia",
    # AFC
    "Iran", "South Korea", "Japan", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # OFC
    "New Zealand",
]

**Filter for Teams Qualified**

We filter the Results dataset to only include matches played between two teams that have qualified for the 2026 FIFA World Cup. This allows the analysis to focus only on relevant teams that will actually participate in the tournament.

In [47]:
# Only includes teams currently qualified for the wolrd cup
results_wc2026 = results_filtered[
    results_filtered['home_team'].isin(world_cup_2026_teams) &
    results_filtered['away_team'].isin(world_cup_2026_teams)
]
results_wc2026

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38395,2015-01-11,Nigeria,Ivory Coast,0,1,Friendly,Abu Dhabi,United Arab Emirates,True
38396,2015-01-11,Tunisia,Algeria,1,1,Friendly,Radès,Tunisia,False
38399,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,Brisbane,Australia,True
...,...,...,...,...,...,...,...,...,...
49062,2026-01-14,Senegal,Egypt,1,0,African Cup of Nations,Tangier,Morocco,True
49063,2026-01-14,Morocco,Nigeria,0,0,African Cup of Nations,Rabat,Morocco,False
49064,2026-01-17,Egypt,Nigeria,0,0,African Cup of Nations,Casablanca,Morocco,True
49065,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,Rabat,Morocco,False


We filter the rankings dataset to only include teams that have qualified for the 2026 FIFA World Cup. This ensures that the rankings data used in the model is relevant to the teams that will actually participate in the tournament.


In [48]:
# Only includes teams currently qualified for the wolrd cup
rankings_wc2026 = rankings_filtered[
    rankings_filtered['country_full'].isin(world_cup_2026_teams)
].sort_values(by=['rank_date', 'rank'], ascending=[False, True])
rankings_wc2026

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
89,Argentina,2024,1.0,ARG,1860.14,1858.00,0,CONMEBOL,2024-06-20
717,France,2024,2.0,FRA,1837.47,1840.59,0,UEFA,2024-06-20
199,Belgium,2024,3.0,BEL,1797.98,1795.23,0,UEFA,2024-06-20
279,Brazil,2024,4.0,BRA,1791.85,1788.65,-1,CONMEBOL,2024-06-20
627,England,2024,5.0,ENG,1787.88,1794.90,1,UEFA,2024-06-20
...,...,...,...,...,...,...,...,...,...
967,Jordan,2015,87.0,JOR,399.00,411.00,5,AFC,2015-12-03
360,Canada,2015,88.0,CAN,388.00,335.00,-14,CONCACAF,2015-12-03
917,Iraq,2015,89.0,IRQ,381.00,392.00,2,AFC,2015-12-03
847,Honduras,2015,101.0,HON,338.00,359.00,6,CONCACAF,2015-12-03


**FIFA Ranking for Each Match Played**

We add FIFA rankings for each team to every match by merging the datasets. This allows us to include each team’s ranking at the time of the match as a feature in the model.

In [49]:
# So the 2 datasets can correlate the countries rank at the time of a match with each other

results_wc2026['year'] = results_wc2026['date'].dt.year
rankings_lookup = rankings_wc2026[['country_full', 'year', 'rank']]


results_wc2026 = (
    results_wc2026.merge(
    rankings_lookup.rename(columns={'country_full': 'home_team', 'rank': 'home_ranking'})
    ).merge(
    rankings_lookup.rename(columns={'country_full': 'away_team', 'rank': 'away_ranking'})
    )
)
results_wc2026 = results_wc2026[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'home_ranking', 'away_ranking']]
results_wc2026


/var/folders/8z/7pzq11j17p17kvqx1snb3zzh0000gn/T/ipykernel_86887/3029627807.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_wc2026['year'] = results_wc2026['date'].dt.year


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0
2,2015-01-11,Tunisia,Algeria,1,1,Friendly,False,40.0,28.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0
...,...,...,...,...,...,...,...,...,...
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0
1021,2024-11-19,Brazil,Uruguay,1,1,FIFA World Cup qualification,False,4.0,14.0


**Sort Tournaments**

We create a function to sort or categorize the values in the tournament column to ensure consistency across the dataset. Since tournament names can vary in format or importance, this function helps standardize them into meaningful groups.

In [50]:

def get_tournament_types(t = 'tournament'):
    
    #Returns the different types of values in the tournaments column
    
    return results_data[t].drop_duplicates().sort_values()

tournaments = get_tournament_types()

tournaments


34280                    ABCS Tournament
4317                       AFC Asian Cup
4213         AFC Asian Cup qualification
29880                  AFC Challenge Cup
31784    AFC Challenge Cup qualification
                      ...               
13117                   West African Cup
5204         Windward Islands Tournament
39936                    World Unity Cup
6144     Zambian Independence Tournament
163                 Évence Coppée Trophy
Name: tournament, Length: 191, dtype: object

**Display Number of Each Unique Tournament**

We display how many times each unique tournament type appears in the dataset to better understand the distribution of matches across different competitions. This helps identify which tournaments are most common and ensures the data is balanced and representative.

In [51]:
results_wc2026['tournament'].value_counts()

tournament
Friendly                                380
FIFA World Cup qualification            228
FIFA World Cup                           77
Copa América                             63
Gold Cup                                 45
UEFA Nations League                      44
CONCACAF Nations League                  37
UEFA Euro                                29
African Cup of Nations                   28
AFC Asian Cup                            23
UEFA Euro qualification                  17
Kirin Challenge Cup                      11
Arab Cup                                  8
African Cup of Nations qualification      8
Confederations Cup                        6
EAFF Championship                         4
Superclásico de las Américas              3
FIFA Series                               3
Gulf Cup                                  2
WAFF Championship                         1
UNCAF Cup                                 1
COSAFA Cup                                1
Kirin Cup            

**Assign Importance to each tournamnet**

We assign importance values to each tournament type to reflect how meaningful different competitions are. More significant tournaments, such as the World Cup or major continental competitions, are given higher importance, while less significant or friendly matches are given lower importance.

We exclude tournament types that appear only a few times in because they do not provide enough data to influence the model. 

These importance values can be adjusted if needed, depending on how well the model performs.

In [52]:

tournament_weights = {
    'FIFA World Cup': 1.00,
    'Copa América' : 0.85,
    'UEFA Euro' : 0.85,
    'AFC Asian Cup': 0.85,
    'African Cup of Nations': 0.85,
    'Gold Cup' : 0.80,
    'Confedertions Cup': 0.75, # Lower because it was discontinuied in 2019 and the final tournamnet was in 2017
    'CONCACAF Nations League': 0.75,
    'UEFA Nations League': 0.75,
    'Superclásico de las Américas': 0.65,
    'FIFA World Cup qualification': 0.65,
    'EAFF Championship': 0.65,
    'Arab Cup': 0.60,
    'UEFA Euro qaulification': 0.55,
    'Copa América qualification': 0.55,
    'WAFF Championship': 0.55,
    'FIFA Series': 0.55,
    'Gulf Cup': 0.55,
    'African Cup of Nations qualification': 0.50,
    'UNCAF Cup': 0.50,
    'COSAFA Cup': 0.50,
    'CAFA Nations Cup': 0.5,
    'Kirin Challenge Cup': 0.35,
    'Kirin Cup': 0.35,
    'Soccer Ashes': 0.30,
    'Friendly': 0.20,
}

results_wc2026_copy = results_wc2026.copy()                             #Copies filtered data 
results_wc2026_copy['tournaments_weight'] = (                           #Creates a new column called 'tournament_weight'
    results_wc2026_copy['tournament'].map(tournament_weights).fillna(0.30) #Assigns 0.30 to any tournament that was not mentioned 
)


**Preparing Match Data for Binary Classification**

Turn the raw match data into something the model can learn from. It creates a target variable where 1 = home team wins and 0 = loses, removes draws so the outcome is only win/loss, adds a feature to compare team strength, and drops rows with missing important values.

In [53]:
results_wc2026["home_win"] = (results_wc2026["home_score"] > results_wc2026["away_score"]).astype(int)
results_wc2026["draw"] = results_wc2026["home_score"] == results_wc2026["away_score"]


model_data = results_wc2026.loc[~results_wc2026["draw"]].copy() #only rows that matches are not a draw


model_data["rank_diff"] = model_data["away_ranking"] - model_data["home_ranking"]#comapre strength


model_data = model_data.dropna(subset=["home_ranking", "away_ranking", "neutral"])
model_data

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking,home_win,draw,rank_diff
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0,1,False,44.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0,1,False,29.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0,0,False,2.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0,0,False,-36.0
5,2015-01-17,Australia,South Korea,0,1,AFC Asian Cup,False,57.0,51.0,0,False,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1016,2024-11-15,Uruguay,Colombia,3,2,FIFA World Cup qualification,False,14.0,12.0,1,False,-2.0
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0,1,False,42.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0,1,False,63.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0,0,False,18.0


**Comparing Win Rate and Ranking Difference**

This plot is showing how the home team’s chance of winning changes based on the ranking difference between the two teams.

In [54]:
win_rank_dif = alt.Chart(model_data,title="Win Rate vs Ranking Difference").mark_line(point=True).encode(
    x=alt.X("rank_diff").bin(True).title("Ranking Difference"),
    y=alt.Y("mean(home_win)").title("Win Rate")
)
win_rank_dif

alt.Chart(...)

From the plot above we can see there is a strong positive correclation this means that ranking difference is a very important feature and strongly related to whether the home team wins.

**Comparing Rankings and Match Outcome**

This plot shows how the home team’s ranking vs the away team’s ranking affects whether the home team wins or loses.

In [69]:
ranking_chart = alt.Chart(model_data, title="Home vs Away Ranking").mark_point(opacity=0.4).encode(
    x=alt.X("home_ranking").title("Home Ranking"),
    y=alt.Y("away_ranking").title("Away Ranking"),
    color=alt.Color("home_win:N")
)
ranking_chart

alt.Chart(...)

From the scatter plot we can see that better ranked teams win more often, so ranking strongly affects whether a team wins or loses.

**Selecting Features and Target**

The features include team rankings, ranking difference, and whether the match is at a neutral venue, while the target is whether the home team wins (1) or loses (0).

In [56]:
X = model_data [[
    "home_ranking",
    "away_ranking",
    "rank_diff",
    "neutral",
]]

y= model_data["home_win"]
X

,home_ranking,away_ranking,rank_diff,neutral
0,45.0,89.0,44.0,True
1,51.0,80.0,29.0,True
3,87.0,89.0,2.0,True
4,89.0,53.0,-36.0,True
5,57.0,51.0,-6.0,False
...,...,...,...,...
1016,14.0,12.0,-2.0,False
1018,11.0,53.0,42.0,False
1019,15.0,78.0,63.0,False
1020,12.0,30.0,18.0,False


**Split Data into Training and Testing Sets**

This splits the data into training and testing sets (75% train, 25% test) while keeping the same proportion of wins and losses and ensuring the results are consistent each time.

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=123,
    stratify=y,
)

**Preprocessing Features**

Scaling the numeric features, keeping the binary feature  unchanged, and fill in any missing numeric values using the median.

In [58]:
numeric_features = ["home_ranking", "away_ranking", "rank_diff"]
binary_features = ["neutral"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        ("bin", "passthrough", binary_features),  
    ],
    remainder="drop",
)

**Tuning KNN Model using Cross-Validation**

Test different K values for the KNN model using cross-validation and select the one with the highest accuracy.

In [59]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier()),
])

param_grid = {
    "model__n_neighbors": range(1, 21, 2),
}

knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

knn_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': range(1, 21, 2)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and

**Train KNN Model with Best Parameters**

In [60]:
knn_model_grid = knn_grid.fit(X_train, y_train)

knn_model_grid

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': range(1, 21, 2)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and

**Cross-Validation Results**

Shows each K value tested along with its average training and testing accuracy to compare model performance.

In [61]:
accuracies_grid = pd.DataFrame(knn_model_grid.cv_results_)

accuracies_grid[[
    "param_model__n_neighbors",
    "mean_test_score",
    "mean_train_score",
]]

,param_model__n_neighbors,mean_test_score,mean_train_score
0,1,0.711125,0.985813
1,3,0.734102,0.855941
2,5,0.748183,0.828903
3,7,0.746460,0.811169
4,9,0.755341,0.795662
5,11,0.769501,0.789011
6,13,0.758881,0.783249
7,15,0.748262,0.777925
8,17,0.739381,0.767287
9,19,0.742936,0.765516


**Accuracy vs K**

This plot shows how the model’s accuracy changes for different K values.

In [71]:
accuracy_versus_k_grid = alt.Chart(accuracies_grid, title="KNN Accuracy vs Number of Neighbors (K)").mark_line(point=True).encode(
    x=alt.X("param_model__n_neighbors")
        .title("Number of Neighbors (K)")
        .scale(zero=False),
    y=alt.Y("mean_test_score")
        .title("Cross-validation Accuracy")
        .scale(zero=False)
)

accuracy_versus_k_grid

alt.Chart(...)

In [72]:
best_k = knn_model_grid.best_params_["model__n_neighbors"]

best_k

11

We can conclude that the K value with highest accuracy is 11.

**Make Predictions and Compare Results**

In [64]:
y_pred = knn_model_grid.predict(X_test)

predictions_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
})

predictions_df.head()


,actual,predicted
0,1,0
1,0,1
2,0,1
3,1,0
4,1,1


**Model Accuracy**

In [65]:
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.6595744680851063

**Example Predictions**

Demonstrate model by making predictions on matches from Canada’s group in the 2026 World Cup to see how well it performs.

In [66]:
example_matches = pd.DataFrame({
    'home_team':    ['Canada', 'Qatar', 'Switzerland', 'Canada', 'Switzerland', 'Bosnia and Herzegovina'],
    'away_team':    ['Bosnia and Herzegovina', 'Switzerland', 'Bosnia and Herzegovina', 'Qatar', 'Canada', 'Qatar'],
    'home_ranking': [30.0,  55.0,  19.0,  30.0,  19.0,  65.0],
    'away_ranking': [65.0,  19.0,  65.0,  55.0,  30.0,  55.0],
    'neutral':      [False, True,  True,  False, False, True],
})

# Match exactly how training data computed it
example_matches['rank_diff'] = example_matches['away_ranking'] - example_matches['home_ranking']

X_examples = example_matches[['home_ranking', 'away_ranking', 'rank_diff', 'neutral']]
example_matches['predicted_home_win'] = knn_grid.predict(X_examples)
example_matches['prediction'] = example_matches['predicted_home_win'].map({1: 'Home Win', 0: 'Away Win'})

example_matches[['home_team', 'away_team', 'home_ranking', 'away_ranking', 'rank_diff', 'prediction']]

,home_team,away_team,home_ranking,away_ranking,rank_diff,prediction
0,Canada,Bosnia and Herzegovina,30.0,65.0,35.0,Home Win
1,Qatar,Switzerland,55.0,19.0,-36.0,Away Win
2,Switzerland,Bosnia and Herzegovina,19.0,65.0,46.0,Home Win
3,Canada,Qatar,30.0,55.0,25.0,Home Win
4,Switzerland,Canada,19.0,30.0,11.0,Home Win
5,Bosnia and Herzegovina,Qatar,65.0,55.0,-10.0,Away Win


In [70]:
points = pd.DataFrame({
    'team': ['Canada', 'Switzerland', 'Qatar', 'Bosnia and Herzegovina'],
    'points': [6, 9, 0, 3]
}).sort_values('points', ascending=False)

Group_Stage_results = alt.Chart(points).mark_bar().encode(
    y=alt.Y('team').title('Nation').sort(None),
    x=alt.X('points').title('Points'),
    color=alt.Color("team").title("Nation"),
).properties(title='Group B Predicted Standings')
Group_Stage_results

alt.Chart(...)

**Discussion & Conclusion**

Our K-nearest neighbors classification model achieved a test accuracy of approximately 66% in predicting match outcomes between 2026 FIFA World Cup qualified nations. Using cross-validation, we determined that K=11 neighbors produced the best balance between overfitting and underfitting.

The results suggest that FIFA ranking-based features, particularly the ranking difference between teams, are moderately useful predictors of match outcomes. The model performed better at predicting home wins (precision: 70%, recall: 81%) than away wins (precision: 55%, recall: 41%), which is consistent with the well-documented home advantage phenomenon in football.

However, a 66% accuracy highlights the inherent difficulty of predicting football outcomes. Our model does not capture important factors such as player availability, recent form, or tournament context, and excluding draws from training simplifies the problem considerably. Future work could explore additional features or alternative classification methods to improve predictive performance.